# Image2GPS Data Collection and Preprocessing

This notebook takes our raw image data from `data/raw/initial_batch`, creates `train` and `validation` folders, randomly sorts the images into those folders, and extracts their metadata into one `metadata.csv` file per folder. We are skipping a local test split for now because the dataset is small and the professor leaderboard/test set is the true held-out test.


In [21]:
# Uncomment and run this cell if your kernel is missing dependencies.
# %pip install exifread pandas

In [22]:
import os
import csv
import json
import random
import shutil

import exifread
import pandas as pd

## Project Paths
Set up the paths for our data

In [23]:
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()

RAW_IMAGE_DIR = os.path.join(REPO_ROOT, "data", "raw", "initial_batch")
DATA_FOLDER = os.path.join(REPO_ROOT, "data", "image2gps_dataset")

SPLITS = ["train", "validation"]
TRAIN_FRACTION = 0.85
VALIDATION_FRACTION = 0.15

RANDOM_SEED = 7

os.makedirs(RAW_IMAGE_DIR, exist_ok=True)
os.makedirs(DATA_FOLDER, exist_ok=True)

print("Raw image folder:", RAW_IMAGE_DIR)
print("Split dataset folder:", DATA_FOLDER)

Raw image folder: /Users/mhedlund/CIS5190/CIS-5190-Final-Project/data/raw/initial_batch
Split dataset folder: /Users/mhedlund/CIS5190/CIS-5190-Final-Project/data/image2gps_dataset


Create split `train` and `validation` subfolders, randomly shuffle images, and create one `metadata.csv` per split.

For now we use an 85/15 train/validation split and rely on the course leaderboard/test set as the real held-out test.


In [24]:
image_files = [
    filename
    for filename in os.listdir(RAW_IMAGE_DIR)
    if filename.lower().endswith((".jpg", ".jpeg")) and os.path.isfile(os.path.join(RAW_IMAGE_DIR, filename))
]

random.seed(RANDOM_SEED)
random.shuffle(image_files)

n_total = len(image_files)
n_train = int(n_total * TRAIN_FRACTION)
split_to_files = {
    "train": image_files[:n_train],
    "validation": image_files[n_train:],
}

# Remove a stale local test split if it exists from an earlier three-way split run.
old_test_dir = os.path.join(DATA_FOLDER, "test")
if os.path.exists(old_test_dir):
    shutil.rmtree(old_test_dir)

for split in SPLITS:
    directory_path = os.path.join(DATA_FOLDER, split)
    os.makedirs(directory_path, exist_ok=True)

    # Clear old image files and old metadata from this generated split folder.
    for filename in os.listdir(directory_path):
        if filename.lower().endswith((".jpg", ".jpeg", ".csv")):
            os.remove(os.path.join(directory_path, filename))

    for filename in split_to_files[split]:
        source_path = os.path.join(RAW_IMAGE_DIR, filename)
        destination_path = os.path.join(directory_path, filename)
        shutil.copy2(source_path, destination_path)

    print(split, len(split_to_files[split]))

train 474
validation 84


# Extracting GPS Information from Images

In [25]:
def get_exif_data(image_path):
    with open(image_path, 'rb') as image_file:
        tags = exifread.process_file(image_file)
    return tags


def export_exif_to_json(exif_data, output_file):
    # Convert tags to a serializable format
    exif_data_serializable = {str(tag): str(value) for tag, value in exif_data.items()}
    with open(output_file, 'w') as json_file:
        json.dump(exif_data_serializable, json_file, indent=4)

In [26]:
# Function to convert GPS coordinates in degrees, minutes, and seconds to decimal degrees
def convert_to_decimal_degrees(value):
    d, m, s = value.values
    return d.num / d.den + (m.num / m.den) / 60 + (s.num / s.den) / 3600

Write one `metadata.csv` file for each train/validation folder.


In [27]:
def create_metadata_csv(train_or_test_or_validation):
    # train_or_test_or_validation could be either train or validation.
    directory_path = f"{DATA_FOLDER}/{train_or_test_or_validation}"
    output_csv = "metadata.csv"
    output_csv = os.path.join(directory_path, output_csv)

    images_with_gps = 0
    images_without_gps = []

    with open(output_csv, mode='w', newline='') as csv_file:
        fieldnames = ['file_name', 'Latitude', 'Longitude']
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)

        # Write the header row
        writer.writeheader()
        for filename in os.listdir(directory_path):
            if not filename.lower().endswith((".jpg", ".jpeg")):
                continue

            if os.path.isfile(os.path.join(directory_path, filename)):
                exif_data = get_exif_data(os.path.join(directory_path, filename))
                if exif_data:
                    gps_latitude = exif_data.get('GPS GPSLatitude', None)
                    gps_latitude_ref = exif_data.get('GPS GPSLatitudeRef', None)
                    gps_longitude = exif_data.get('GPS GPSLongitude', None)
                    gps_longitude_ref = exif_data.get('GPS GPSLongitudeRef', None)
                    if gps_latitude and gps_longitude:
                        # Convert latitude and longitude to decimal degrees
                        latitude = convert_to_decimal_degrees(gps_latitude)
                        longitude = convert_to_decimal_degrees(gps_longitude)

                        # Adjust for N/S and E/W reference
                        if gps_latitude_ref.values[0] == 'S':
                            latitude = -latitude
                        if gps_longitude_ref.values[0] == 'W':
                            longitude = -longitude

                        writer.writerow({'file_name': filename, 'Latitude': latitude, 'Longitude': longitude})
                        images_with_gps += 1
                    else:
                        images_without_gps.append(filename)
                else:
                    images_without_gps.append(filename)

    return output_csv, images_with_gps, images_without_gps

## Create `metadata.csv` Files

In [28]:
metadata_results = {}

for train_or_test_or_validation in SPLITS:
    output_csv, images_with_gps, images_without_gps = create_metadata_csv(train_or_test_or_validation)
    metadata_results[train_or_test_or_validation] = {
        "output_csv": output_csv,
        "images_with_gps": images_with_gps,
        "images_without_gps": images_without_gps,
    }
    print(train_or_test_or_validation, "metadata:", output_csv)
    print("  images with GPS:", images_with_gps)
    print("  images without GPS:", len(images_without_gps))

train metadata: /Users/mhedlund/CIS5190/CIS-5190-Final-Project/data/image2gps_dataset/train/metadata.csv
  images with GPS: 474
  images without GPS: 0
validation metadata: /Users/mhedlund/CIS5190/CIS-5190-Final-Project/data/image2gps_dataset/validation/metadata.csv
  images with GPS: 84
  images without GPS: 0


## Quick Data Inspection

In [29]:
metadata_frames = []

for split in SPLITS:
    csv_path = metadata_results[split]["output_csv"]
    df = pd.read_csv(csv_path)
    df["split"] = split
    metadata_frames.append(df)

metadata_df = pd.concat(metadata_frames, ignore_index=True)

print("Total labeled images:", len(metadata_df))
display(metadata_df.groupby("split").size().rename("labeled_images"))
display(metadata_df[["Latitude", "Longitude"]].describe())
display(metadata_df.head())

Total labeled images: 558


split
train         474
validation     84
Name: labeled_images, dtype: int64

,Latitude,Longitude
count,558.000000,558.000000
mean,39.951698,-75.191512
std,0.000626,0.000589
min,39.950447,-75.192475
25%,39.951289,-75.191922
50%,39.951625,-75.191632
75%,39.952286,-75.191188
max,39.952899,-75.190156


,file_name,Latitude,Longitude,split
0,IMG_20260410_173246007_HDR.jpg,39.950932,-75.191260,train
1,IMG_20260410_172442736.jpg,39.951814,-75.191704,train
2,IMG_1875.JPG,39.950744,-75.191361,train
3,IMG_1861.JPG,39.951144,-75.190850,train
4,IMG_1849.JPG,39.951378,-75.190619,train


In [30]:
for split in SPLITS:
    missing = metadata_results[split]["images_without_gps"]
    print(split, "images without GPS:", len(missing))
    print(missing[:20])

train images without GPS: 0
[]
validation images without GPS: 0
[]
